# 2회차 미션 노트북

오늘 실습은 미션 여섯 개로 진행함. 자세한 배경·힌트·기록표는 『2회차 미션 워크북』(PDF)에 있슴

- 셀은 위에서부터 차례로 Shift + Enter로 실행함
- `!python experiments/...` 셀은 옵션을 바꿔 가며 여러 번 돌려 봄. 무엇을 바꿨을 때 결과가 달라지는지가 핵심임
- 연결이 끊기면 0번 셀부터 다시 실행함. 결과 파일은 맨 아래 셀로 내려받음

| 미션 | 의뢰 | 쓰는 것 | 시간 |
|---|---|---|---|
| M0 | 입사 시험: 직선 하나로 안 나뉘는 문제 | `xor_demo.py` | 10분 |
| M1 | 기억력 감사: 긴 문장 앞부분을 잊는 이유 | `seq_limits.py`, seq2seq 학습 | 20분 |
| M2 | 어텐션 탐정: 모델은 문장을 어떻게 읽나 | `experiments/s02/self_attention.py`, `attention_probe.py` | 25분 |
| M3 | 토큰 요금 견적: 챗봇 한 달 요금 | `token_cost.py` | 12분 |
| M4 | 다음 단어 맞히기: 속담 이벤트에 써도 되나 | `experiments/s01/next_token.py`, `next_word_game.py` | 13분 |
| M5 | 어제 그 리뷰를 트랜스포머로, 학습 없는 분류기 경연 | `llm_sentiment.py`, `llm_classify.py` | 23분 |
| M6 | (과제) FAQ 검색기 재기 | `faq_search.py` | 집에서 |

## 0. 준비
- 런타임 유형은 T4 GPU 권장(런타임 → 런타임 유형 변경). CPU로도 모든 셀이 돎
- 아래 두 셀을 실행해 판정이 '통과'인지 봄. '확인 필요'가 나오면 TA를 부름

In [ ]:
import os, sys, shutil, subprocess, threading
from IPython.display import Image, Markdown, display

REPO = "https://github.com/elliediaz/llm26-course.git"
if os.path.isfile("../src/config.py"):          # 저장소 안에서 열었을 때
    HOME = os.path.abspath("..")
elif os.path.isdir("/workspace/course"):        # 실습 서버
    HOME = "/workspace/course"
else:                                           # 코랩
    HOME = "/content/course" if os.path.isdir("/content") else os.path.abspath("llm26-course")
    if not os.path.isdir(HOME):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, HOME], check=True)
    else:
        subprocess.run(["git", "-C", HOME, "pull", "-q"])
os.chdir(HOME)
print("작업 폴더:", os.getcwd())
os.makedirs("results/fig", exist_ok=True)
os.makedirs("prompts", exist_ok=True)


def show(*paths, width=760):
    for p in paths:
        display(Image(p, width=width)) if os.path.exists(p) else print("아직 없음:", p)


def tail(path, n=8):
    if not os.path.exists(path):
        print("아직 없음:", path)
        return
    lines = open(path, encoding="utf-8", errors="replace").read().splitlines()
    print("\n".join(lines[-n:]))


def use_ref():
    # 학습이 늦을 때 1회차 참고 결과를 복사함(내 결과는 덮어쓰지 않음)
    n = 0
    for f in sorted(os.listdir("ref/s01")):
        dst = {".json": "results", ".png": "results/fig", ".pt": "runs"}.get(os.path.splitext(f)[1])
        if dst and not os.path.exists(os.path.join(dst, f)):
            os.makedirs(dst, exist_ok=True)
            shutil.copy(os.path.join("ref/s01", f), dst)
            n += 1
    print(f"참고 결과 {n}개 복사(이미 있던 파일은 그대로 둠)")

In [ ]:
%pip install -q koreanize-matplotlib sentence-transformers
!python src/check_gpu.py

## M0. 입사 시험: XOR를 나눠라
- 단층 · 층만 쌓기 · 층 + 활성화 함수, 세 모델로 AND·OR·XOR를 학습함(약 10초)
- 기록: XOR 줄의 정확도 세 개, `--hidden 2`일 때 시드마다 달라지는 값

In [ ]:
!python src/xor_demo.py
show("results/fig/xor_boundary.png", width=620)

In [ ]:
for seed in [0, 1, 3]:
    !python src/xor_demo.py --hidden 2 --seed {seed} | grep XOR
# !python src/xor_demo.py --act sigmoid

확인 값
- XOR 줄: 단층 0.50, 2층·활성화 X 0.50, 2층·ReLU 1.00
- `--hidden 2`: 시드 0은 0.75, 시드 3은 1.00. 뉴런이 적으면 시작값 운에 따라 갈림

## M1. 기억력 감사: 앞부분을 잊는 이유
의뢰: 고객사 챗봇이 긴 문의의 앞부분을 자꾸 놓친다고 함. 어제 쓴 RNN·LSTM이 어디까지 기억하는지 재고, 같은 과제를 트랜스포머 층으로 바꾸면 버티는지 확인

1) 기억 길이 시험: 숫자 여러 개 중 맨 앞 숫자를 맞힘(찍으면 0.1). 같은 데이터·같은 1,000스텝으로 몸통만 RNN, LSTM, 트랜스포머 층 1개로 바꿔 학습함(코랩 CPU 약 1분 30초)

In [ ]:
!python src/seq_limits.py
show("results/fig/seq_limits.png")

2) 휴식 전 걸어 두기: 한글 금액을 숫자로 바꾸는 seq2seq 세 구조를 2분씩 학습함(약 7분)
- 쉬는 동안 탭을 닫지않음. 도는 동안 다른 학습 셀은 실행하지 않음

In [ ]:
def _run_jobs():
    with open("results/log_seq2seq.txt", "w", encoding="utf-8") as f:
        subprocess.run([sys.executable, "src/tiny_seq2seq_train.py", "--arch", "all", "--minutes", "2"],
                       stdout=f, stderr=subprocess.STDOUT)
    open("results/bg_done.txt", "w").write("done")


if os.path.exists("results/bg_done.txt"):
    os.remove("results/bg_done.txt")
threading.Thread(target=_run_jobs, daemon=True).start()
print("뒤에서 학습 시작. 진행 확인은 다음 셀")

In [ ]:
tail("results/log_seq2seq.txt", 4)
print("\n모두 끝남" if os.path.exists("results/bg_done.txt") else "\n아직 학습 중")

3) 휴식 후: 세 구조의 결과와 어텐션 그림. 학습이 덜 끝났으면 `use_ref()`로 참고 결과를 받아 진행함

In [ ]:
tail("results/log_seq2seq.txt", 6)
# use_ref()

In [ ]:
!python src/viz_attention.py tiny --arch gru_attn --text "삼억 칠천이백만 오천 원"
show("results/fig/attn_tiny_gru_attn.png")

확인 값
- 기억 길이(정확도, 길이 5 · 10 · 20 · 40): RNN 1.00 · 1.00 · 0.20 · 0.09, LSTM 1.00 · 1.00 · 0.10 · 0.10, 트랜스포머 층 1.00 · 1.00 · 1.00 · 1.00
- 시간은 거꾸로임: 길이 40에서 RNN 6.4초, 트랜스포머 층 19.8초(CPU). 점수표가 길이×길이라서 길어질수록 계산이 늘어남. GPU에서는 이 계산을 한꺼번에 처리함
- 더 해 보기: `!python src/seq_limits.py --lengths 20 40 80` (약 2분)
- seq2seq: 17자 이상에서 `gru`는 약 0.5로 떨어지고 `gru_attn`은 약 1.00
- 그림: 출력 숫자마다 입력의 어느 글자를 봤는지가 대각선으로 보임

## M2. 어텐션 탐정: 모델은 문장을 어떻게 읽나
의뢰: "모델이 '그것'이 무엇인지 아는가?"라는 질문에 근거를 들어 답하는 짧은 보고서

1) Self-Attention을 직접 계산함. 마스크를 빼거나 √d 나누기를 빼면 주의 가중치가 어떻게 바뀌는지 봄

In [ ]:
!python experiments/s02/self_attention.py

In [ ]:
!python experiments/s02/self_attention.py --no-mask                # 미래 토큰도 봄(인코더 방식)
!python experiments/s02/self_attention.py --dim 512 --no-scale     # 나누기를 빼면 한 칸에 쏠림

1-1) 블록 하나의 모양 확인: 작은 GPT(4층)의 텐서 모양과 파라미터 수를 Qwen3와 나란히 봄(약 15초)

In [ ]:
!python src/mini_transformer.py --info

2) 같은 낱말, 다른 뜻: 과일 배 · 타는 배 · 몸의 배가 모델 안에서 다른 벡터가 되는지 봄(약 20초)

In [ ]:
!python src/attention_probe.py --polysemy
show("results/fig/polysemy.png")

3) '그것'은 무엇인가: 지시어 문장 12개에서 층별 어텐션을 재고, 모델에게 직접도 물어봄(약 20초)

In [ ]:
!python src/attention_probe.py
show("results/fig/attention_probe.png")

4) 내 문장으로 수사: 따옴표 안을 바꿔 다시 실행 함. 후보는 쉼표로 구분

In [ ]:
!python src/attention_probe.py --text "엄마가 딸에게 전화했다. 그녀는 집에 늦게 온다고 했다." --target 그녀 --cands 엄마,딸
show("results/fig/attention_probe_text.png", width=620)

확인 값
- 2-T1: 가중치 첫 행 1.00, 둘째 행 0.45·0.55, 대각선 위쪽 0
- 블록 모양: 들어갈 때와 나올 때 모두 [B, T, d], 점수표는 (1, 4, 16, 16). 작은 GPT 880,256개, Qwen3-0.6B 약 6억 개
- 다의어: 0층(임베딩)에서는 '배'끼리 유사도가 모두 1.00. 마지막 층에서는 같은 뜻 0.93, 다른 뜻 0.84로 벌어짐
- 지시어: big·small 쌍에서 '지시어 자리' 값이 똑같음(trophy 쪽 0.41, 뒤 단어를 아직 못 봄). 묻기 정답률 0.50(0.6B 모델의 한계)

## M3. 토큰 요금 견적: 챗봇 한 달 요금
의뢰: 배달앱 고객사가 상담 챗봇의 월 요금 견적을 요청함. 모델(토크나이저)마다 요금이 얼마나 달라지는지 비교

1) 표본 문장 8종 × 토크나이저 4종(처음 실행 때 토크나이저를 내려받아 약 30초)

In [ ]:
!python src/token_cost.py
show("results/fig/token_cost.png")

2) 쪼개진 모양 보기: 내가 고른 문장을 넣어 봄. `·`은 한 글자가 여러 토큰으로 갈라진 조각

In [ ]:
!python src/token_cost.py --text "점메추 해 줘. 오늘은 갓생 살 거라서" "결제했는데 아이템이 안 들어왔어요"

3) 견적 조건 바꾸기: 하루 요청 수, 답변 길이, 단가를 우리 서비스에 맞게 바꿔 봄

In [ ]:
!python src/token_cost.py --traffic game --daily 50000 --answer-ratio 5 --price 3000 | tail -8

확인 값
- 한국어 일상 문장 글자/토큰: A.X 3.06 · Qwen3 1.12 · KURE 1.62 · GPT-2 0.44
- 영어 문장은 네 토크나이저가 모두 같음(19토큰). 차이는 한국어에서 남
- 기본 조건 월 요금: A.X 약 7.7만 원, Qwen3 약 13.9만 원, GPT-2 약 41.9만 원

## M4. 다음 단어 맞히기
의뢰: 마케팅팀이 '속담 이어 말하기' 이벤트에 LLM을 쓰자고 함. 써도 되는지 판정

1) 다음 토큰 분류기 직접 보기(처음 실행 때 모델을 내려받아 1~2분 더 걸림). 문장을 바꿔 1등 확률이 갈리는 단계를 찾음

In [ ]:
!python experiments/s01/next_token.py --steps 10

2) 먼저 맞혀 보기: 아래 앞부분 세 개의 다음 단어를 종이에 적은 뒤 실행함

In [ ]:
!python src/next_word_game.py --prefix "가는 말이 고와야" "The early bird catches the" "오늘 점심 메뉴는"

3) 속담 50개 채점: 대화용(Instruct)과 원본(Base) 비교(약 2분)

In [ ]:
!python src/next_word_game.py --model both | tail -12
show("results/fig/next_word_game.png", width=560)

4) 온도 바꿔 보기: 같은 앞부분에서 다섯 번 뽑음

In [ ]:
!python src/next_word_game.py --prefix "우리 팀 프로젝트 이름은" --temp 1.0 --samples 5
# !python src/next_word_game.py --prefix "우리 팀 프로젝트 이름은" --temp 0.3 --samples 5

확인 값
- T3: 어휘 크기 151,936, 확률 합 1.0
- 속담: 영어는 대부분 맞히고 한국어는 거의 못 맞힘. 학습 데이터에 들어 있던 양의 차이
- 온도 1.0은 매번 다른 이름, 0.3은 거의 같은 이름

## M5. 어제 그 리뷰를 트랜스포머로, 그리고 학습 없는 분류기
어제(1회차) LSTM은 영화 리뷰 3만 건으로 학습해 0.80을 냈음. 오늘은 같은 리뷰를 대량의 글로 미리 학습해 둔 트랜스포머(Qwen3-0.6B)로, 가중치를 하나도 바꾸지 않고 분류함

① 필수: 어제 그 리뷰(NSMC 60건). 먼저 예시 없이, 다음은 예시 4개를 붙여 봄(각 약 1분)

In [ ]:
!python src/llm_sentiment.py

In [ ]:
!python src/llm_sentiment.py --shots 4
show("results/fig/llm_sentiment.png", width=560)
# !python src/llm_sentiment.py --shots 4 --model Qwen/Qwen3-0.6B-Base

확인 값(어제 LSTM과 비교)

| 방법 | 학습 | 정확도 | 길이별(1~20 · 21~40 · 41~80 · 81자 이상) |
|---|---|---|---|
| 어제 LSTM | 리뷰 3만 건 · 2에폭 | 0.80 | 0.80 · 0.80 · 0.82 · 0.76 |
| Qwen3-0.6B 예시 0개 | 없음 | 0.50(모두 '긍정') | - |
| Qwen3-0.6B 예시 4개 | 없음 | 0.68 | 0.70 · 0.70 · 0.69 · 0.50 |
| Qwen3-0.6B-Base 예시 4개 | 없음 | 0.78 | - |

- 60건이라 길이 칸마다 몇 건뿐임. 81자 이상 칸은 특히 흔들림이 큼
- 생각해 볼 것: 학습 0건으로 0.68~0.78이 나온 것은 사전학습 덕분. 남은 차이는 큰 모델(3·4회차)과 우리 데이터로 다시 학습(7회차)으로 메움

② 선택·경연: 같은 방법을 다른 고객사 데이터로. 학습할 시간과 GPU가 없으므로 가중치는 그대로 두고 프롬프트만으로 정확도를 올림

| 과제 | 고객사(가상) | 라벨 | 찍기 수준 |
|---|---|---|---|
| `delivery` | 배달앱 | 맛 · 배달 · 양 · 포장 · 칭찬 | 0.20 |
| `smishing` | 통신사 보안팀 | 정상 · 스미싱 | 0.50 |
| `news` | 포털 뉴스팀 | IT과학 · 경제 · 사회 · 생활문화 · 스포츠 | 0.20 |
| `game` | 게임사 운영팀 | 결제 · 밸런스 · 버그 · 이벤트 · 제재 | 0.20 |
| `nsmc` | (1회차) 영화 리뷰 | 부정 · 긍정 | 0.50 |

규칙
- 개발셋(`--split dev`, 기본값)으로 몇 번이고 고쳐 봄. 시험셋(`--split test`)은 마지막에 한 번만 돌림
- 고칠 수 있는 손잡이는 다섯 개: 지시문(`--prompt`), 예시 수(`--shots`), 라벨 단어(`--words`), 보정(`--calibrate`), 모델(`--model base`)
- 마지막 줄 '제출 줄'을 팀 기록표와 공유 시트에 붙임

1) 우리 팀 과제 고르기

In [ ]:
TASK = "delivery"      # delivery · smishing · news · game · nsmc 중 하나
TAG = "팀이름"          # 리더보드에 남길 이름
!python src/llm_classify.py --list

2) 기준 점수: 아무것도 고치지 않은 상태

In [ ]:
!python src/llm_classify.py --task {TASK} --show-prompt --tag {TAG}_기준
show(f"results/fig/llm_classify_{TASK}.png", width=420)

3) 손잡이 돌리기: 한 번에 하나씩 바꾸고 결과를 기록표에 적음

In [ ]:
!python src/llm_classify.py --task {TASK} --shots 5 --tag {TAG}_예시5

In [ ]:
!python src/llm_classify.py --task {TASK} --shots 5 --calibrate --tag {TAG}_예시5_보정

4) 지시문 고치기: 아래 파일 내용을 우리 과제에 맞게 고쳐 씀. 라벨 정의, 헷갈리는 경우의 기준을 적으면 효과가 큼

In [ ]:
%%writefile prompts/my_prompt.txt
배달 음식 리뷰를 읽고 고객이 무엇에 대해 말하는지 하나만 고름.
맛: 맛, 간, 익힘, 냄새에 대한 불만
배달: 배달 시간, 기사, 도착 과정에 대한 불만
양: 양이 적거나 빠진 품목이 있음
포장: 포장이 새거나 더럽거나 이물질이 나옴
칭찬: 만족하거나 다시 주문하겠다는 내용

In [ ]:
!python src/llm_classify.py --task {TASK} --prompt prompts/my_prompt.txt --shots 5 --calibrate --tag {TAG}_지시문1

5) 라벨 단어와 모델 바꾸기(선택): 라벨 순서대로 다른 단어를 줌. Base 모델은 대화용으로 다듬기 전의 원본

In [ ]:
# !python src/llm_classify.py --task delivery --shots 5 --calibrate --words "맛없음,늦음,부족,새거나더러움,만족"
# !python src/llm_classify.py --task {TASK} --shots 5 --calibrate --model base --tag {TAG}_base
!python src/llm_classify.py --task {TASK} --shots 5 --calibrate --text "사장님 감사해요 근데 국물이 다 샜어요"

6) 최종 제출: 개발셋에서 가장 좋았던 설정 그대로 시험셋을 한 번만 돌림

In [ ]:
!python src/llm_classify.py --task {TASK} --split test --prompt prompts/my_prompt.txt --shots 5 --calibrate --tag {TAG}_최종

In [ ]:
import pandas as pd
pd.read_csv("results/leaderboard.csv").tail(12)

확인 값(Qwen3-0.6B, 시험셋)

| 과제 | 예시 0 | 예시 0 + 보정 | 예시 5 | 예시 5 + 보정 | Base · 예시 5 + 보정 |
|---|---|---|---|---|---|
| delivery | 0.20 | 0.26 | 0.56 | 0.64 | 0.62 |
| news | 0.20 | 0.33 | 0.45 | 0.56 | 0.56 |
| game | 0.24 | 0.39 | 0.43 | 0.61 | 0.71 |
| smishing(예시 2) | 0.53 | 0.58 | 0.98 | 0.98 | 0.78 |

- 위 4)의 예시 지시문 + 예시 5 + 보정(delivery): 개발셋 0.80, 시험셋 0.70
- 예시 0개일 때 한 라벨로만 답하는 쏠림이 보이면 보정부터 켬
- 이 값을 넘기면 지시문이 효과를 낸 것

## M6. (과제) 내 질문 10건으로 FAQ 검색기 재기
- 아래 셀의 질문 10건을 고침(표현 바꾸기 5, 오타 2, 은어 2, 무관 질문 1)
- `gold_faq_id`에 정답 번호를 씀(`data/G/faq.json`). 무관 질문은 `null`
- 10건 중 8건 이상 맞히면 완료

In [ ]:
%%writefile my_queries.json
[
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "표현 변형",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "오타",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "오타",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "은어",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "은어",
  "query": "여기에 질문을 씀",
  "gold_faq_id": "F__"
 },
 {
  "variant": "무관 질문",
  "query": "여기에 FAQ와 상관없는 질문을 씀",
  "gold_faq_id": null
 }
]

In [ ]:
!python src/faq_search.py --queries my_queries.json
# !python src/faq_search.py --method tfidf --queries my_queries.json

## 결과 확인과 내려받기
- 오늘 남겨야 할 파일(M0·M1·M2·M3·M5)이 모두 있는지 확인함. '누락'이 있으면 해당 미션 셀을 다시 실행함
- 코랩은 연결이 끝나면 파일이 사라지므로 수업을 마치기 전에 내려받음

In [ ]:
!python src/resume_check.py --session 2 --quick

In [ ]:
shutil.make_archive("s02_results", "zip", ".", "results")
try:
    from google.colab import files
    files.download("s02_results.zip")
except ImportError:
    print("저장 위치:", os.path.abspath("s02_results.zip"))

## 더 해 보기(선택)
- 생성 설정: greedy · 온도 · Top-k · Top-p별로 30번 뽑아 서로 다른 토큰 수를 비교함. `--settings`로 조합을 바꿔 봄
- 나만의 데이터: `data/s02/README.md` 구조로 라벨당 15건을 써서 `--data`로 돌려 봄

In [ ]:
!python experiments/s02/sampling.py

In [ ]:
!python experiments/s02/sampling.py --settings t=0 t=0.7 t=0.7,p=0.5 t=2.0 t=2.0,k=3 --draws 50

## 문제가 생기면
| 증상 | 조치 |
|---|---|
| `No such file or directory` | 0번 셀부터 다시 실행 |
| 연결이 끊김 | 다시 연결하고 0번 셀부터 실행. M1 학습은 다시 걸거나 `use_ref()` |
| GPU를 쓸 수 없다는 안내 | CPU로 그대로 진행 |
| 셀이 끝나지 않음 | 셀 왼쪽 ■를 누르고 TA를 부름 |
| `Rate limited` 문구 | 1~2분 기다리면 이어짐 |
| `--words 는 라벨 수만큼` | 라벨 순서대로 같은 개수를 쉼표로 적음 |